# Training-free guidance (TFG) on CIFAR-100

Baseline 2 of 3: the condition enters through the **sampling trajectory**. An
unconditional diffusion model is steered at every step by the gradient of a classifier
evaluated on the predicted clean image. Nothing is trained: the diffusion model is the
one already trained here (run with its null label), and the classifier is a pretrained
CIFAR-100 ResNet from `torch.hub`.

**Paper:** Ye, Lin, Han, Xu, Liu, Liang, Ma, Zou, Ermon.
*TFG: Unified Training-Free Guidance for Diffusion Models.* NeurIPS 2024.
[arXiv:2409.15761](https://arxiv.org/abs/2409.15761)

**Official code exists** and was read:
[YWolfeee/Training-Free-Guidance](https://github.com/YWolfeee/Training-Free-Guidance),
`methods/tfg.py`. `src/guidance/baselines/tfg.py` re-expresses `guide_step` against our
DDIM sampler, with their schedules and their CIFAR-10 label-guidance hyperparameters as
defaults. Its header lists what was kept and what changed.

**Two classifiers, deliberately different.** ResNet-56 guides; VGG-16 scores. If one
network did both, "condition accuracy" would measure how well guidance games its own
objective. A guidance method has not been shown to work because its own objective went up.

Run top to bottom. On Colab set **Runtime > Change runtime type > GPU** first.

## How this notebook implements TFG

**Notation.** $x_k$ is the noisy image at diffusion step $k$ ($K = 1000$; the network receives
$t = k/K$), $k'$ the next step of the $N = 50$-step DDIM schedule, $\bar\alpha_k$ the cosine
schedule, $\varepsilon_\theta$ the frozen U-Net, $\varnothing$ the null label. $y$ is the target
class and $p_r(y \mid x)$ a frozen pretrained classifier on clean images.

**The paper's idea.** Steer an *unconditional* diffusion model with any differentiable objective
on clean data, without training anything: at every step, estimate the clean image, score it,
and move along the gradient. TFG shows that earlier training-free methods are special cases of
two terms: a *variance* term, the gradient with respect to the noisy state taken through the
denoiser, and a *mean* term, gradient steps taken on the clean estimate itself. Both act on a
smoothed objective, and both have a schedule over the trajectory. Per step $k \to k'$
(Algorithm 1):

$$\hat x_0 = \operatorname{clip}\!\Big(\frac{x_k - \sqrt{1-\bar\alpha_k}\;\varepsilon_\theta(x_k, k, \varnothing)}{\sqrt{\bar\alpha_k}}\Big),
\qquad
\log \tilde f(x) = \log \frac{1}{n_\varepsilon}\sum_{i=1}^{n_\varepsilon} \exp f\big(x + \sigma^{\mathrm s}_k\,\delta_i\big),\quad f(x) = \log p_r(y \mid x)$$

$$\Delta_k = \rho_k\, \nabla_{x_k} \log \tilde f\big(\hat x_0(x_k)\big)
\qquad\text{(variance guidance; the gradient goes through } \varepsilon_\theta)$$

$$\hat x_0^{(j+1)} = \hat x_0^{(j)} + \mu_k\, \nabla \log \tilde f\big(\hat x_0^{(j)}\big),\ \ j = 0,\dots,N_{\text{iter}}-1,
\qquad \Delta_0 = \hat x_0^{(N_{\text{iter}})} - \hat x_0
\qquad\text{(mean guidance)}$$

$$x_{k'} = \underbrace{\sqrt{\bar\alpha_{k'}}\,\hat x_0 + \sqrt{1-\bar\alpha_{k'}}\,\hat\varepsilon}_{\text{plain DDIM step, } \hat\varepsilon \text{ recomputed from the clipped } \hat x_0}
\;+\; \frac{\Delta_k}{\sqrt{\bar\alpha_k / \bar\alpha_{k'}}} \;+\; \sqrt{\bar\alpha_{k'}}\;\Delta_0$$

Schedules are theirs: $\rho_k = \rho\, a_k / \bar a$ and $\mu_k = \mu\, a_k / \bar a$ with
$a_k = \bar\alpha_k / \bar\alpha_{k'}$ and $\bar a$ its mean over the trajectory, so the mean
strength equals the $\rho$ and $\mu$ you set; $\sigma^{\mathrm s}_k = \sigma^{\mathrm s}\sqrt{1-\bar\alpha_k}$.
(The paper writes the smoothing scale $\sigma_t$; it is $\sigma^{\mathrm s}$ here so that $\sigma$
is not overloaded.)

**How it is used here.** The unconditional denoiser is our class-conditional U-Net run with the
null label. The objective is $\log p_r(y \mid \hat x_0)$ from a pretrained CIFAR-100 ResNet-56
after converting the estimate into the classifier's input normalisation. The hyperparameters are
their CIFAR-10 label-guidance script, unchanged. The DDIM step inside the guided sampler is the
same step the rest of this repository uses, and the first cell after setup asserts that at
$\rho = \mu = 0$ the guided sampler reproduces plain DDIM exactly. Nothing is trained. Each
guided step costs one denoiser forward, one backward, and $N_{\text{iter}} + 1$ classifier
evaluations; the notebook reports the counts rather than a single NFE.

| In the paper / official code | In this notebook |
|---|---|
| Alg. 1, clean estimate $\hat x_0$, clamped | `TFG._predict_x0` in `tfg.py` |
| Alg. 1, variance guidance $\Delta_k$, gradient through the denoiser | the `rho` block of `TFG.sample` |
| Alg. 1, mean guidance, $N_{\text{iter}}$ steps on the clean estimate | the `iter_steps` loop of `TFG.sample` |
| Alg. 1, the update $x_{k'}$; `_predict_x_prev_from_zero` recomputes $\hat\varepsilon$ from the clipped $\hat x_0$ | `TFG.sample`; `ddim_sample` here uses the same convention, and *Set up TFG* asserts the two agree at $\rho = \mu = 0$ |
| $\log\tilde f$, `tilde_get_guidance` | `TFG._log_f_tilde` |
| schedules `get_rho`, `get_mu`, `get_std` | `TFG._strength`, `TFG._std` |
| `ImageLabelGuidance`: the objective is the classifier's $\log p(y \mid \hat x_0)$ | `make_log_p`, with `to_clf` converting into the classifier's input space |
| `scripts/cifar10_label.sh`: $\rho = 1$, $\mu = 0.25$, $\sigma^{\mathrm s} = 10^{-3}$, $n_\varepsilon = 1$, $N_{\text{iter}} = 4$, one recurrence, 50 steps | the `Cfg` defaults |
| an unconditional diffusion model | `eps_uncond`: our class-conditional model run with the null label |
| not in TFG: a second classifier for scoring, the wrong-target row, paired generators, the strength sweep | *Sample*, *Condition accuracy*, *Guidance-strength sweep*: our evaluation protocol |

**What the paper did that this notebook does not.** Their sampler is stochastic DDIM
($\eta = 1$); ours is deterministic ($\eta = 0$) to match the other arms, and the $\eta$ term is
implemented if you want theirs. Their diffusion model is an unconditional pretrained CIFAR-10
DDPM; ours is our own conditional U-Net standing in as unconditional through the null label.
They searched hyperparameters per task; we take their CIFAR-10 values as they are, so these
settings are not tuned for CIFAR-100. Six deviations are listed in the header of
`src/guidance/baselines/tfg.py`.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("datasets",):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    # Inside a checkout (Jupyter on a server, cwd somewhere under the repo): walk up to its src/.
    from pathlib import Path as _P
    _src = next((p / "src" for p in [_P.cwd(), *_P.cwd().parents] if (p / "src" / "guidance").is_dir()), None)
    if _src is None:                                   # Colab: clone the repo next to the notebook
        if not os.path.exists("discrete-generative-models-proj-1"):
            subprocess.run(["git", "clone", "-q",
                            "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                           check=True)
        _src = "discrete-generative-models-proj-1/src"
    sys.path.insert(0, str(_src))
    print("guidance package from:", _src)

import torch
from guidance.baselines.tfg import TFG, TFGConfig

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA)
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Config

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict, fields
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 10                 # training / sampling randomness
    split_seed: int = 10           # validation split; fixed so multi-seed runs share one split
    batch_size: int = 64
    base_epochs: int = 30
    lr_base: float = 2e-4
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 50         # sampling steps; TFG's CIFAR-10 script also uses 50
    reuse_base: bool = True        # load base.pt if present instead of retraining
    limit: int = 0                 # >0 truncates the training set, for smoke tests
    # TFG -- defaults are the official CIFAR-10 label-guidance script
    rho: float = 1.0
    mu: float = 0.25
    sigma: float = 0.001
    eps_bsz: int = 1
    iter_steps: int = 4
    recur_steps: int = 1
    eta: float = 0.0               # 0 matches our other arms (deterministic DDIM); their script used 1.0
    guide_clf: str = "cifar100_resnet56"     # steers the sampler
    eval_clf: str = "cifar100_vgg16_bn"      # scores the result; must differ from guide_clf

cfg = Cfg()

# Any field can be overridden from the environment, e.g. CFG_BASE_EPOCHS=1 CFG_LIMIT=5000, so
# headless runs (nbconvert, a server, a different GPU) never need a cell edited.
for _f in fields(Cfg):
    _v = os.environ.get(f"CFG_{_f.name.upper()}")
    if _v is not None:
        _cur = getattr(cfg, _f.name)
        setattr(cfg, _f.name, tuple(float(t) for t in _v.split(",")) if isinstance(_cur, tuple)
                else type(_cur)(_v) if not isinstance(_cur, bool) else _v.lower() in ("1", "true", "yes"))
        print(f"cfg.{_f.name} <- {getattr(cfg, _f.name)!r} (from CFG_{_f.name.upper()})")
assert cfg.guide_clf != cfg.eval_clf, "guide and eval classifiers must be different networks"
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

From `src/guidance/data/cifar.py`. Pixels in `[-1, 1]`; 50 images per class held out for
validation. Local copy first, then the HuggingFace CDN.

In [ ]:
from guidance.data.cifar import make_loaders

train_loader, val_loader, meta = make_loaders(
    batch_size=cfg.batch_size, seed=cfg.split_seed, limit=cfg.limit)

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), f"[{float(xb.min()):.2f}, {float(xb.max()):.2f}]",
      "| cond", tuple(cb.shape))
print("train batches", len(train_loader), "| val batches", len(val_loader))

## Pretrained classifiers

Both from [chenyaofo/pytorch-cifar-models](https://github.com/chenyaofo/pytorch-cifar-models)
via `torch.hub`, trained on CIFAR-100 at native 32×32. Neither is retrained.

They expect inputs normalised with CIFAR-100's channel mean and std; our diffusion model
works in `[-1, 1]`. `to_clf` converts. Getting this wrong makes the classifier see garbage
and the guidance silently do nothing — so the cell checks each classifier's accuracy on
real validation images before anything else runs.

In [ ]:
CIFAR_MEAN = torch.tensor([0.5071, 0.4865, 0.4409], device=DEV).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2673, 0.2564, 0.2762], device=DEV).view(1, 3, 1, 1)

def to_clf(x):
    '''[-1, 1] diffusion space -> the classifiers' normalised input space.

    The clamp is part of the guidance objective: during mean refinement the clean estimate
    can drift outside the pixel range, and there the objective's gradient is exactly zero.
    That is deliberate -- out-of-range pixels are not images -- but it means the objective
    TFG optimises is log p(y | clamp(x0)), not log p(y | x0). Saturation is reported below.
    '''
    return ((x.clamp(-1, 1) + 1) / 2 - CIFAR_MEAN) / CIFAR_STD

def load_clf(name):
    m = torch.hub.load("chenyaofo/pytorch-cifar-models", name, pretrained=True, verbose=False)
    m = m.to(DEV).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m

guide_clf = load_clf(cfg.guide_clf)
eval_clf = load_clf(cfg.eval_clf)

@torch.no_grad()
def accuracy(clf, loader):
    '''Full pass. The validation set is class-ordered, so a partial pass sees few classes.'''
    hit = tot = 0
    for x, y in loader:
        hit += (clf(to_clf(x.to(DEV))).argmax(1) == y.to(DEV)).sum().item(); tot += len(y)
    return hit / tot

# This is a plumbing check, not a held-out measurement: both classifiers were trained on
# CIFAR-100's training set, and our validation images are drawn from that same set. High
# accuracy here means to_clf is right. It says nothing about generalisation.
for name, clf in [(cfg.guide_clf, guide_clf), (cfg.eval_clf, eval_clf)]:
    acc = accuracy(clf, val_loader)
    print(f"{name:>22}: {acc:.1%} top-1 on {len(val_loader.dataset)} real images (all 100 classes)")
    assert acc > 0.5, f"{name} is not seeing sensible inputs; check to_clf"

## Backbone

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10_000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([args.cos(), args.sin()], dim=-1)


def conv_block(cin, cout, groups=8):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.GroupNorm(groups, cout), nn.SiLU(),
        nn.Conv2d(cout, cout, 3, padding=1), nn.GroupNorm(groups, cout), nn.SiLU(),
    )


class UNet(nn.Module):
    '''Class-conditional U-Net. The label enters as a per-channel bias after block 1.'''

    def __init__(self, c=64, n_classes=100, tdim=128):
        super().__init__()
        self.tdim, self.null = tdim, n_classes
        self.t_mlp = nn.Sequential(nn.Linear(tdim, c), nn.SiLU(), nn.Linear(c, c))
        self.label = nn.Embedding(n_classes + 1, c)      # +1 = null token
        self.e1, self.e2 = conv_block(3, c), conv_block(c, 2 * c)
        self.mid = conv_block(2 * c, 4 * c)
        self.d2, self.d1 = conv_block(6 * c, 2 * c), conv_block(3 * c, c)
        self.out = nn.Conv2d(c, 3, 1)
        self.pool = nn.MaxPool2d(2)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        bias = (self.t_mlp(timestep_embedding(t, self.tdim))
                + self.label(cond_ids))[:, :, None, None]
        s1 = self.e1(x) + bias
        s2 = self.e2(self.pool(s1))
        h = self.mid(self.pool(s2))
        h = self.d2(torch.cat([F.interpolate(h, size=s2.shape[-2:]), s2], 1))
        h = self.d1(torch.cat([F.interpolate(h, size=s1.shape[-2:]), s1], 1))
        return self.out(h)


def make_backbone():
    return UNet(64, meta["n_conditions"])

torch.manual_seed(cfg.seed)      # model init and training randomness follow `seed`, not the split
net = make_backbone().to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

## Diffusion — cosine schedule, epsilon prediction, DDIM sampler

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

ABAR = cosine_alpha_bar(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]
CLIP_X0 = 1.0


def add_noise(x0, gen=None):
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device, generator=gen)
    eps = torch.randn(x0.shape, device=x0.device, generator=gen)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, x_init=None, eta=None, generator=None):
    '''Plain deterministic DDIM, TFG's convention: clip x0, then RECOMPUTE eps from the
    clipped x0 before stepping. Keeping the original eps is a different update whenever
    the clamp engages; the two can differ by O(1) on real images. The TFG sampler in
    src/ reduces to exactly this at rho = mu = 0, and the cell below asserts it.'''
    clip_x0 = CLIP_X0 if clip_x0 is None else clip_x0
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    eta = cfg.eta if eta is None else eta
    x = torch.randn(n, *shape, device=DEV, generator=generator) if x_init is None else x_init.clone()
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
        if clip_x0:
            x0 = x0.clamp(-clip_x0, clip_x0)
            eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()      # recompute from the clipped x0
        # eta = 0 is deterministic DDIM; eta = 1 is ancestral DDPM (DDIM paper eq. 16), with fresh
        # noise at every step drawn from `generator` so paired rows stay paired.
        sigma = eta * ((1 - ab_prev) / (1 - ab) * (1 - ab / ab_prev)).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        if eta > 0:
            x = x + sigma * torch.randn(x.shape, device=DEV, generator=generator)
    return x

## Base model — train, or load `base.pt`

Identical to the other arms. Upload `base.pt` from a previous run to skip the eight
minutes; every arm then sits on the same frozen weights.

In [ ]:
VAL_SEED = cfg.seed + 999


def run_epochs(model, params, n_epochs, lr, tag, train_fn, val_fn, ckpt):
    opt = torch.optim.AdamW(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1))
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            loss = train_fn(x, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()
        val = validate(model, val_fn)
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best,
                     "seconds": time.time() - t0})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


@torch.no_grad()
def validate(model, val_fn):
    model.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    tot = num = 0
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        tot += val_fn(x, y, gen).item() * x.shape[0]; num += x.shape[0]
    return tot / num


def base_train(x, y):
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(y)), eps)


def base_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(net(xk, k.float() / cfg.K, y), eps)


CKPT_META = {"split_seed": cfg.split_seed, "n_conditions": meta["n_conditions"],
             "shape": list(meta["shape"]), "K": cfg.K, "schedule": "cosine",
             "backbone_params": sum(p.numel() for p in net.parameters())}

if cfg.reuse_base and os.path.exists("base.pt"):
    ck = torch.load("base.pt", map_location=DEV)
    saved = ck.get("meta")
    if saved is None:
        print("WARNING: base.pt carries no setup metadata (older run); split and schedule "
              "cannot be verified.")
    else:
        mismatch = {k: (saved.get(k), v) for k, v in CKPT_META.items() if saved.get(k) != v}
        assert not mismatch, f"base.pt was trained under a different setup: {mismatch}"
    net.load_state_dict(ck["state"])
    base_hist = [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val"], "best": True}]
    print(f"loaded base.pt (epoch {ck['epoch']}, val {ck['val']:.4f}); skipping base training")
else:
    base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                           "base", base_train, base_val, "base.pt")
    ck = torch.load("base.pt", map_location=DEV); ck["meta"] = CKPT_META; torch.save(ck, "base.pt")

# Whichever branch ran, sample from the BEST checkpoint on disk, not the last epoch left in
# `net`. Without this a fresh run and a reused-checkpoint run would use different weights
# while the manifest reported the same best validation loss for both.
ck = torch.load("base.pt", map_location=DEV)
net.load_state_dict(ck["state"])
base_frozen = net.eval()
for p in base_frozen.parameters():
    p.requires_grad_(False)
print(f"sampling from base.pt epoch {ck['epoch']} (val {ck['val']:.4f})")

## Set up TFG

TFG expects an **unconditional** denoiser. Ours is class-conditional with a null label, so
`eps_uncond` runs it with the null label throughout — the label path is off, and the
classifier gradient is the only route by which the class enters.

The objective is `log p(y | x̂₀)` from the frozen guide classifier, evaluated on the
predicted clean image after conversion to the classifier's input space. That is what
their `ImageLabelGuidance` returns.

First check: at `rho = mu = 0` the TFG sampler must reproduce plain DDIM to numerical
precision. If it does not, nothing downstream can be trusted.

In [ ]:
def eps_uncond(x, t):
    # TFG assumes an unconditional denoiser. Ours is class-conditional with a null label, so the
    # null label stands in for "unconditional" and the classifier gradient is the only way in.
    return base_frozen(x, t, torch.full((x.shape[0],), NULL_ID, device=x.device, dtype=torch.long))


def make_log_p(y):
    '''log p(y | x0) under the guide classifier, for a fixed batch of targets y.
    This is the ImageLabelGuidance objective: the quantity Algorithm 1 climbs at every step.'''
    def log_p(x0):
        logits = guide_clf(to_clf(x0))
        yy = y.repeat(x0.shape[0] // y.shape[0])          # smoothing may stack eps_bsz copies
        return F.log_softmax(logits, dim=-1).gather(1, yy[:, None]).squeeze(1)
    return log_p


tfg_cfg = TFGConfig(rho=cfg.rho, mu=cfg.mu, sigma=cfg.sigma, eps_bsz=cfg.eps_bsz,
                    iter_steps=cfg.iter_steps, recur_steps=cfg.recur_steps,
                    clip_x0=CLIP_X0, eta=cfg.eta)

# sanity: guidance off == plain DDIM, same starting noise
g = torch.Generator(device=DEV).manual_seed(0)
x0_probe = torch.randn(8, *meta["shape"], device=DEV, generator=g)
y_probe = torch.arange(8, device=DEV)
off = TFG(eps_uncond, make_log_p(y_probe), ABAR, TFGConfig(rho=0, mu=0, iter_steps=0, clip_x0=CLIP_X0))
a = off.sample(8, meta["shape"], 10, x_init=x0_probe)
b = ddim_sample(eps_uncond, 8, meta["shape"], 10, x_init=x0_probe, eta=0.0)   # deterministic on both sides
assert torch.allclose(a, b, atol=1e-4), (a - b).abs().max().item()
assert off.stats.objective_evals == 0 and off.stats.denoiser_backward == 0
print("rho = mu = 0 reproduces plain DDIM exactly")
print("TFG config:", asdict(tfg_cfg))

## Sample — four rows from the same starting noise

| row | what | label path | guidance |
|---|---|---|---|
| **uncond** | base, null label | off | none |
| **label** | base, true label | on | none — the reference for what the label is worth |
| **TFG** | base, null label | off | classifier gradient toward `y` |
| **TFG, wrong y** | base, null label | off | classifier gradient toward `y + 1` |

The last row is the diagnostic: if TFG works, its samples should score as `y + 1`, not
`y`. Guidance that ignores its target would leave rows 3 and 4 identical.

Every row is timed. TFG costs a denoiser backward pass and several classifier passes per
step, so it is not one number cheaper or dearer — report the breakdown.

**Paired means paired.** Same `X0` is not enough: TFG draws smoothing noise inside the
loop (`sigma > 0`), and with recurrence it would draw re-noising too. Every TFG call in a
comparison gets its own generator seeded identically, so the *only* thing that differs
between rows is the target. Two per class, all 100 classes, so accuracy is not a
statement about classes 0–63.

In [ ]:
N = 2 * meta["n_conditions"]                       # every class twice
y_eval = torch.arange(N, device=DEV) % meta["n_conditions"]
y_wrong = (y_eval + 1) % meta["n_conditions"]
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 1)
X0 = torch.randn(N, *meta["shape"], device=DEV, generator=g)

def paired_gen():
    '''Fresh generator, same seed: identical internal randomness for every matched call.'''
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 1234)

def timed(fn):
    if DEV.type == "cuda": torch.cuda.synchronize()
    t0 = time.time(); out = fn()
    if DEV.type == "cuda": torch.cuda.synchronize()
    return out, (time.time() - t0) / N

rows = {}
rows["uncond"], t_uncond = timed(lambda: ddim_sample(eps_uncond, N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
rows["label"], t_label = timed(lambda: ddim_sample(lambda x, t: base_frozen(x, t, y_eval), N, meta["shape"],
                                                   cfg.sample_steps, x_init=X0, generator=paired_gen()))
tfg = TFG(eps_uncond, make_log_p(y_eval), ABAR, tfg_cfg)    # Algorithm 1, their CIFAR-10 settings
rows["tfg"], t_tfg = timed(lambda: tfg.sample(N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
tfg_stats = tfg.stats
tfg_w = TFG(eps_uncond, make_log_p(y_wrong), ABAR, tfg_cfg)
rows["tfg_wrong"], _ = timed(lambda: tfg_w.sample(N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))

# How often the guided clean estimate hit the clamp -- the objective's gradient is zero there.
SATURATION = {k: (v.abs() >= 0.999).float().mean().item() for k, v in rows.items()}

print(f"{'row':>10} {'s / sample':>11}   denoiser fwd / bwd   objective calls   saturated px")
print(f"{'uncond':>10} {t_uncond:>11.3f}   {cfg.sample_steps:>4} / {0:<4}          0          {SATURATION['uncond']:6.1%}")
print(f"{'label':>10} {t_label:>11.3f}   {cfg.sample_steps:>4} / {0:<4}          0          {SATURATION['label']:6.1%}")
print(f"{'tfg':>10} {t_tfg:>11.3f}   {tfg_stats.denoiser_evals:>4} / {tfg_stats.denoiser_backward:<4}          "
      f"{tfg_stats.objective_evals}        {SATURATION['tfg']:6.1%}")

## Condition accuracy — scored by the *other* classifier

`eval_clf` never touched the sampler. Its accuracy on each row is the target metric.
For the wrong-target row the number reported is accuracy against `y + 1`: high means
guidance followed its target.

In [ ]:
@torch.no_grad()
def acc_of(x, y):
    return (eval_clf(to_clf(x)).argmax(1) == y).float().mean().item()

ACC = {
    "uncond": acc_of(rows["uncond"], y_eval),
    "label": acc_of(rows["label"], y_eval),
    "tfg": acc_of(rows["tfg"], y_eval),
    "tfg_wrong_vs_target": acc_of(rows["tfg_wrong"], y_wrong),
    "tfg_wrong_vs_original": acc_of(rows["tfg_wrong"], y_eval),
}
for k, v in ACC.items():
    print(f"{k:>24}: {v:6.1%}")
print()
print(f"chance = {1 / meta['n_conditions']:.1%}.")
print("If guidance follows its target, 'tfg' sits well above 'uncond' and 'tfg_wrong_vs_target' is "
      "comparably high. 'tfg_wrong_vs_original' is informative but not a pass/fail: it can sit "
      "below chance, or above it through systematic class confusion, without guidance being wrong.")
if ACC["tfg"] < ACC["uncond"] + 0.05:
    print("little steering at these settings: try larger rho/mu or more iter_steps, and re-check to_clf.")
print()
print("Accuracy alone rewards unrealistic images even with a separate evaluator. Read it with FID.")

## Samples

In [ ]:
import matplotlib.pyplot as plt

def short(i):
    return meta["cond_text"][int(i)].replace("a photo of a ", "")

show = 8
fig, axes = plt.subplots(4, show, figsize=(2 * show, 8.6))
for r, (name, s, ys) in enumerate([("uncond", rows["uncond"], y_eval), ("label", rows["label"], y_eval),
                                   ("TFG", rows["tfg"], y_eval), ("TFG, wrong y", rows["tfg_wrong"], y_wrong)]):
    for j in range(show):
        axes[r, j].imshow(((s[j].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
        if r in (2, 3):
            axes[r, j].set_title(short(ys[j]), fontsize=8)
    axes[r, 0].set_ylabel(name, fontsize=10)
plt.tight_layout(); plt.show()

## Guidance-strength sweep

TFG's `rho` and `mu` are the knobs. A short sweep at fixed `mu/rho` ratio shows the
accuracy-vs-time trade-off. This is the analogue of a classifier-free-guidance strength
sweep, and it is what a learned guidance controller has to beat: the **best point on this
curve**, not the default.

This is a first pass, not TFG's own hyperparameter search, and it is run on validation
targets. Pick a setting here, then report it with fresh sampling seeds — do not report the
maximum of the sweep as the result.

In [ ]:
SWEEP = [0.0, 0.5, 1.0, 2.0, 4.0]
sweep_rows = []
for scale in SWEEP:
    c = TFGConfig(rho=cfg.rho * scale, mu=cfg.mu * scale, sigma=cfg.sigma, eps_bsz=cfg.eps_bsz,
                  iter_steps=cfg.iter_steps if scale > 0 else 0, recur_steps=cfg.recur_steps,
                  clip_x0=CLIP_X0, eta=cfg.eta)
    s, dt = timed(lambda: TFG(eps_uncond, make_log_p(y_eval), ABAR, c).sample(
        N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
    sweep_rows.append({"scale": scale, "rho": c.rho, "mu": c.mu, "acc": acc_of(s, y_eval), "s_per_sample": dt})
    print(f"scale {scale:>4}  rho {c.rho:>5.2f}  mu {c.mu:>5.3f}  acc {sweep_rows[-1]['acc']:6.1%}  {dt:.3f}s/sample")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot([r["scale"] for r in sweep_rows], [r["acc"] for r in sweep_rows], marker="o")
ax.axhline(ACC["label"], ls="--", c="grey", lw=1, label=f"label-conditioned base ({ACC['label']:.1%})")
ax.axhline(1 / meta["n_conditions"], ls=":", c="grey", lw=1, label="chance")
ax.set_xlabel("guidance scale (x default rho, mu)"); ax.set_ylabel(f"accuracy ({cfg.eval_clf})")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## FID (optional, needs a GPU)

Set `n_fid = 10000` for a reportable number. Reports the unconditional base and TFG at the
default strength. Real images are the 5000-image validation split.

In [ ]:
n_fid = int(os.environ.get("N_FID", 0))   # 10000 for a reportable number; 0 skips this cell. Headless: N_FID=10000

FID_SCORES = {}

if n_fid:
    import importlib.util, subprocess, sys
    for pkg, mod in [("torchmetrics", "torchmetrics"), ("torch-fidelity", "torch_fidelity")]:
        if importlib.util.find_spec(mod) is None:
            print(f"installing {pkg} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    from torchmetrics.image.fid import FrechetInceptionDistance

    def to_uint8(x):
        return ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)

    def sample_for(which, y):
        if which == "uncond":
            return ddim_sample(eps_uncond, len(y), meta["shape"], cfg.sample_steps)
        return TFG(eps_uncond, make_log_p(y), ABAR, tfg_cfg).sample(len(y), meta["shape"], cfg.sample_steps)

    for which in ("uncond", "tfg"):
        fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEV)
        n_real = 0
        for x, _ in val_loader:
            fid.update(to_uint8(x.to(DEV)), real=True); n_real += len(x)
        torch.manual_seed(cfg.seed)
        for start in range(0, n_fid, 100):
            y = torch.arange(start, min(start + 100, n_fid), device=DEV) % meta["n_conditions"]
            fid.update(to_uint8(sample_for(which, y)), real=False)
        FID_SCORES[f"{which}_fid"] = float(fid.compute())
        FID_SCORES["fid_n_real"] = n_real
        FID_SCORES["fid_n_generated"] = n_fid
        print(f"{which} FID {FID_SCORES[f'{which}_fid']:.2f}   (real {n_real}, generated {n_fid})")
    if min(FID_SCORES["fid_n_real"], n_fid) < 10000:
        print()
        print("NOTE: FID is biased upward below ~10k samples. Report both counts.")
else:
    print("FID skipped (n_fid = 0)")

## Manifest

In [ ]:
report = {
    "method": "TFG training-free guidance (arXiv:2409.15761), pretrained CIFAR-100 classifier",
    "modality": "images",
    "code_version": GIT_SHA,
    "config": asdict(cfg),
    "tfg_config": asdict(tfg_cfg),
    "guide_classifier": cfg.guide_clf,
    "eval_classifier": cfg.eval_clf,
    "label_path": False,
    "accuracy": ACC,
    "n_eval_samples": N,
    "saturated_pixel_fraction": SATURATION,
    "sweep": sweep_rows,
    "seconds_per_sample": {"uncond": t_uncond, "label": t_label, "tfg": t_tfg},
    "tfg_cost_per_sample": {"denoiser_forward": tfg_stats.denoiser_evals,
                            "denoiser_backward": tfg_stats.denoiser_backward,
                            "objective_evals": tfg_stats.objective_evals},
    "plain_ddim_cost_per_sample": {"denoiser_forward": cfg.sample_steps},
    "base_best_val": min(h["val"] for h in base_hist),
    "checkpoint_meta": CKPT_META,
    "trainable_params": 0,
    "frozen_backbone": sum(p.numel() for p in base_frozen.parameters()),
    "nfe_sample_steps": cfg.sample_steps,
    "train_seconds": {"base": sum(h.get("seconds", 0) for h in base_hist)},
    "hardware": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
    "device": str(DEV),
}
report.update(globals().get("FID_SCORES", {}))
print(json.dumps(report, indent=2))
with open("manifest_tfg_images.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] Both classifiers passed the real-image check (>50%), so `to_clf` is right. That check
      is on images the classifiers trained on; it is plumbing, not a held-out number
- [ ] `tfg_wrong_vs_target` is comparably high to `tfg` — guidance follows its target
- [ ] FID with `n_fid = 10000`, read together with accuracy, never accuracy alone
- [ ] Three **sampling** seeds on one checkpoint, mean +/- std, stated as such. Independently
      trained backbones are a different, more expensive experiment; say which you did
- [ ] Guidance settings chosen on the validation sweep, then reported with fresh seeds
- [ ] Time per sample and the denoiser/objective call counts reported next to accuracy;
      TFG is not one NFE number
- [ ] Saturation fraction reported; the objective is log p(y | clamp(x0))
- [ ] Described as TFG's `guide_step` **re-expressed** against our sampler with their CIFAR-10
      hyperparameters as starting settings (not tuned for CIFAR-100) and `eta = 0`, on a
      null-label conditional model standing in for an unconditional one